# RipsNet Shape Classification

Benchmarking **12 neural network architectures** on **6 point cloud classification tasks**.

We compare:
- **Baselines**: PersNet, RipsPointNet, ScalarInputMLP, ScalarDistanceDeepSet
- **Tensor Field Networks**: TFN, GTTFNv2, Hierarchical TFN, Stochastic TFN
- **O(n)-equivariant**: OnEquivariant TFN, Attention TFN, Relaxed TFN, Hybrid TFN

Datasets include 2D concentric circles (clean and noisy) and 4 variants of 3D synthetic shapes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, glob, os
from collections import defaultdict

## 1. Dataset Overview

Six classification tasks with increasing difficulty:

| Dataset | Classes | Dimension | Task |
|---------|---------|-----------|------|
| circles | 3 | 2D | Classify concentric circle patterns |
| circles_noisy | 3 | 2D | Same, with outlier noise points |
| shapes3d_topology | 4 | 3D | Distinguish shapes by topology (sphere vs torus etc.) |
| shapes3d_geometry | 4 | 3D | Distinguish shapes by geometry (size, radius) |
| shapes3d_complex | 6 | 3D | Mixed topological + geometric variation |
| shapes3d_8way | 8 | 3D | 8-class fine-grained classification |

In [ ]:
# Generate and visualize sample point clouds
from datasets.utils import create_multiple_circles
from datasets.shapes3d import generate_dataset, DATASET_CONFIGS

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# 2D circles
data_clean, labels_clean = create_multiple_circles(3, 600, noisy=False, N_noise=200)
data_noisy, labels_noisy = create_multiple_circles(3, 600, noisy=True, N_noise=200)

for i, label in enumerate(sorted(set(labels_clean))):
    idx = labels_clean.index(label)
    pc = data_clean[idx]
    axes[0, i].scatter(pc[:, 0], pc[:, 1], s=1, alpha=0.5)
    axes[0, i].set_title(f'Clean: {label}')
    axes[0, i].set_aspect('equal')
axes[0, 3].axis('off')

# 3D shapes (project to 2D for visualization)
for i, (name, cfg) in enumerate(list(DATASET_CONFIGS.items())[:4]):
    n_per = 30
    data, labels, _, _, classes = generate_dataset(name, n_per, 10, 600, noise_sigma=0.0, seed=42)
    idx = labels.index(classes[0])
    pc = data[idx]
    axes[1, i].scatter(pc[:, 0], pc[:, 2], s=1, alpha=0.5)
    axes[1, i].set_title(f'3D: {name}')
    axes[1, i].set_aspect('equal')

plt.tight_layout()
plt.suptitle('Sample Point Clouds', y=1.02, fontsize=14)
plt.show()

## 2. Load Results

All experiments produce JSON files in `shape/results/`. Each file contains validation accuracy, test accuracy (clean), noisy test accuracy, and hyperparameters.

In [ ]:
results_dir = 'shape/results'
files = sorted(glob.glob(os.path.join(results_dir, 'shape_*.json')))
print(f'Found {len(files)} result files')

records = []
for f in files:
    with open(f) as fh:
        d = json.load(fh)
    records.append(d)

df = pd.DataFrame(records)
print(f'\nModels: {df["model"].nunique()}')
print(f'Datasets: {df["dataset"].nunique()}')
print(f'Trials per combo: {df.groupby(["dataset","model"]).size().describe()}')
df.head(10)

## 3. Results Summary

Mean and standard deviation of validation accuracy across 3 trials.

In [ ]:
# Compute summary statistics
summary = df.groupby(['dataset', 'model']).agg(
    val_mean=('best_val_accuracy', 'mean'),
    val_std=('best_val_accuracy', 'std'),
    clean_mean=('clean_accuracy', 'mean'),
    clean_std=('clean_accuracy', 'std'),
    noisy_mean=('noisy_accuracy', 'mean'),
    noisy_std=('noisy_accuracy', 'std'),
    n_trials=('trial', 'count'),
).reset_index()

# Pivot for clean accuracy
pivot = summary.pivot_table(
    index='model', columns='dataset',
    values='clean_mean', aggfunc='first'
)
pivot = pivot.round(3)
print('Clean Test Accuracy (mean over 3 trials):')
pivot

## 4. Visualize Results

In [ ]:
MODEL_SHORT = {
    'PersNet': 'PersNet',
    'RipsPointNet': 'RipsPointNet',
    'ScalarInputMLP': 'Scalar MLP',
    'ScalarDistanceDeepSet': 'Scalar DeepSet',
    'TensorFieldNetwork': 'TFN',
    'GTTensorFieldNetworkV2': 'GT-TFN v2',
    'HierarchicalTensorFieldNetwork': 'Hier. TFN',
    'StochasticTensorFieldNetwork': 'Stoch. TFN',
    'OnEquivariantTensorFieldNetwork': 'O(n)-TFN',
    'AttentionTensorFieldNetwork': 'Attn. TFN',
    'RelaxedOnEquivariantTensorFieldNetwork': 'Relaxed TFN',
    'HybridOnEquivariantTensorFieldNetwork': 'Hybrid TFN',
}

datasets = sorted(df['dataset'].unique())
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, ds in zip(axes.flat, datasets):
    ds_data = summary[summary['dataset'] == ds].sort_values('clean_mean', ascending=True)
    models = [MODEL_SHORT.get(m, m) for m in ds_data['model']]
    means = ds_data['clean_mean'].values * 100
    stds = ds_data['clean_std'].values * 100
    stds = np.nan_to_num(stds, nan=0)
    
    colors = ['#2196F3' if 'TFN' in m or 'GT' in m else '#FF9800' if 'Point' in m else '#4CAF50' for m in models]
    ax.barh(models, means, xerr=stds, color=colors, edgecolor='white', height=0.6)
    ax.set_xlim(0, 105)
    ax.set_xlabel('Clean Accuracy (%)')
    ax.set_title(ds, fontsize=12, fontweight='bold')
    ax.axvline(x=100, color='gray', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.suptitle('Clean Test Accuracy by Dataset (mean ± std over 3 trials)', y=1.02, fontsize=14)
plt.show()

## 5. Noisy Robustness

Compare clean vs noisy accuracy to measure robustness to input corruption.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, ds in zip(axes, ['circles', 'shapes3d_topology', 'shapes3d_8way']):
    ds_data = summary[summary['dataset'] == ds].sort_values('clean_mean')
    models = [MODEL_SHORT.get(m, m)[:12] for m in ds_data['model']]
    x = np.arange(len(models))
    
    ax.barh(x - 0.15, ds_data['clean_mean'].values * 100, 0.3, label='Clean', color='#2196F3')
    ax.barh(x + 0.15, ds_data['noisy_mean'].values * 100, 0.3, label='Noisy', color='#FF5722')
    ax.set_yticks(x)
    ax.set_yticklabels(models)
    ax.set_xlim(0, 105)
    ax.set_xlabel('Accuracy (%)')
    ax.set_title(ds, fontweight='bold')
    ax.legend(loc='lower right')

plt.tight_layout()
plt.suptitle('Clean vs Noisy Accuracy', y=1.02)
plt.show()

## 6. Key Findings

Based on 132 completed experiments (50 epochs, 300 train / 100 test, 3 trials each):

**Clean accuracy (no noise):**
- TFN variants (TFN, GTTFNv2, Hierarchical, Stochastic) hit **100%** on every clean dataset
- PersNet, RipsPointNet, and ScalarInputMLP reach **99–100%** on all clean tasks
- ScalarDistanceDeepSet reaches **93–100%** — strong but slightly below other baselines

**Robustness to noise (50% corrupted points):**
- PersNet is the **most robust**, e.g. 84.9% on circles_noisy vs 59–74% for TFN variants
- ScalarInputMLP is also robust (82.3% circles_noisy) — simpler models generalize better
- TFN variants **drop sharply** under noise: 100% → 68–74% on circles_noisy
- ScalarDistanceDeepSet is sensitive to noise on 2D: 84% circles, 58% circles_noisy

**3D datasets (shapes3d_complex, shapes3d_8way):**
- All models perform well on clean 3D data (93–100%)
- Under noise, the same pattern holds: PointNet variants > ScalarInput > TFN variants

**Remaining:** Hybrid and Relaxed models still training (submitted with 3-day SLURM limit)

## 7. Reproducing

```bash
# Single experiment
python shape/train_shape.py circles TensorFieldNetwork 50 0

# All experiments (on cluster)
cd shape && python check_missing.py

# Consolidate results
python shape/consolidate_results.py
```